## Movie Recommendation

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
movies = pd.read_csv("../data/raw/movies.csv")
ratings = pd.read_csv("../data/raw/ratings.csv")

print("Dataset Loaded Successfully")

## Data Information

In [ ]:
print(f"Movies Shape : {movies.shape}")
print(f"Ratings Shape: {ratings.shape}")

## Movies Dataset

In [ ]:
movies.head()

In [ ]:
movies.info()

In [ ]:
print(f"Rows    : {movies.shape[0]}")
print(f"Columns : {movies.shape[1]}")

In [ ]:
movies.describe(include="all").T

## Ratings Dataset

In [ ]:
ratings.head()

In [ ]:
ratings.info()

In [ ]:
print(f"Rows    : {ratings.shape[0]}")
print(f"Columns : {ratings.shape[1]}")

In [ ]:
ratings.describe(include="all").T

## Missing Values

In [ ]:
print("Movies Missing Values")
print(movies.isnull().sum())

print()

print("Ratings Missing Values")
print(ratings.isnull().sum())

## Duplicate Records

In [ ]:
print("Movies Duplicates :", movies.duplicated().sum())
print("Ratings Duplicates:", ratings.duplicated().sum())

## Number of Movies

In [ ]:
movies["movieId"].nunique()

## Number of Users

In [ ]:
ratings["userId"].nunique()

## Number of Ratings

In [ ]:
len(ratings)

## Average Rating

In [ ]:
ratings["rating"].mean()

## Ratings Distribution

In [ ]:
plt.figure(figsize=(8, 5))

ratings["rating"].value_counts().sort_index().plot(kind="bar")

plt.title("Ratings Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")

plt.show()

## Genre Distribution

In [ ]:
genres = (
    movies["genres"]
    .str.split("|")
    .explode()
)

genres.value_counts().plot(
    kind="bar",
    figsize=(12, 5)
)

plt.title("Genre Distribution")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.show()

## Top 10 Most Rated Movies

In [ ]:
movie_ratings = ratings.groupby("movieId").size().reset_index(name="rating_count")

movie_ratings = movie_ratings.merge(
    movies,
    on="movieId",
)

movie_ratings.sort_values(
    by="rating_count",
    ascending=False,
).head(10)

## Highest Rated Movies (Minimum 100 Ratings)

In [ ]:
movie_stats = ratings.groupby("movieId").agg(
    average_rating=("rating", "mean"),
    rating_count=("rating", "count"),
)

movie_stats = movie_stats.merge(
    movies,
    on="movieId",
)

movie_stats[
    movie_stats["rating_count"] >= 100
].sort_values(
    by="average_rating",
    ascending=False,
).head(10)

## Merge Movies and Ratings

In [ ]:
movie_data = ratings.merge(
    movies,
    on="movieId",
)

movie_data.head()

In [ ]:
movie_data.columns

In [ ]:
movie_data.shape

In [ ]:
movie_rating_count = (
    movie_data
    .groupby("title")["rating"]
    .count()
    .reset_index(name="rating_count")
)

movie_rating_count.head()

In [ ]:
popular_movies = movie_rating_count[
    movie_rating_count["rating_count"] >= 50
]

popular_movies.shape

In [ ]:
movie_data = movie_data.merge(
    popular_movies,
    on="title",
)

movie_data.head()

## Create Movie × User Matrixmovie_user_matrix.head()

In [ ]:
movie_user_matrix = movie_data.pivot_table(
    index="title",
    columns="userId",
    values="rating",
).fillna(0)

In [ ]:
movie_user_matrix.head()

In [ ]:
movie_user_matrix.shape

## Convert to Sparse Matrix

In [ ]:
from scipy.sparse import csr_matrix

movie_sparse_matrix = csr_matrix(
    movie_user_matrix.values
)

## Train the KNN Model

In [ ]:
from sklearn.neighbors import NearestNeighbors

knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=6,
)

knn_model.fit(movie_sparse_matrix)

## Test the Model

In [ ]:
movie_index = movie_user_matrix.index.get_loc("Toy Story (1995)")

distances, indices = knn_model.kneighbors(
    movie_sparse_matrix[movie_index]
)

In [ ]:
for index in indices.flatten():
    print(movie_user_matrix.index[index])